In [1]:
import sys
print(sys.executable)


c:\Office365\Moon Capital\Quantitative Intern - Documents\Pengxin(Grace)\1. Target_trace_variable\venv\Scripts\python.exe


In [5]:
import sys
print(sys.executable)


c:\Users\pli\AppData\Local\Programs\Python\Python313\python.exe


In [ ]:
import pandas as pd
import openpyxl
import xlwings
print("All packages imported successfully")

All packages imported successfully


In [ ]:
import re
import datetime
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter, range_boundaries

# === Load workbooks ===
WORKBOOK_FILE = "Selected Specific Workbook File.xlsx"
TARGET_SHEET = "Interested Sheet"
TARGET_CELL = "Interested Cell"
BASELINE_COMPARISON_CELL = "Target Comparison"

manual_today_date = datetime.date(2025, 6, 24)
start_row_for_test = 100

wb_formulas = load_workbook(WORKBOOK_FILE, data_only=False)
wb_values = load_workbook(WORKBOOK_FILE, data_only=True)

In [ ]:
import re
import datetime
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter, range_boundaries

# === Load workbooks ===
WORKBOOK_FILE = "sample_forecast_model.xlsx"
TARGET_SHEET = "Target Forecast Sheet"
TARGET_CELL = "Target Forecast Cell"
BASELINE_COMPARISON_CELL = "Target Comparison"

manual_today_date = datetime.date(2025, 6, 24)
start_row_for_test = 100

wb_formulas = load_workbook(WORKBOOK_FILE, data_only=False)
wb_values = load_workbook(WORKBOOK_FILE, data_only=True)

In [21]:
def extract_numeric_constants(formula):
    if not formula or not formula.startswith("="):
        return []

    # Remove quoted strings (e.g., text literals in formulas)
    cleaned = re.sub(r'"[^"]*"', '', formula)

    # Tokenize the formula into pieces split by operators, parentheses, commas, spaces
    tokens = re.split(r'[\+\-\*/\^\(\),\s]+', cleaned)

    constants = []
    for token in tokens:
        token = token.strip()
        # Skip if it's a cell reference like A1, AA123, etc.
        if re.match(r'^[A-Za-z]{1,3}[0-9]{1,5}$', token):
            continue
        try:
            constants.append(float(token))
        except ValueError:
            pass  # ignore non-numeric tokens
    return constants

def resolve_label_with_hierarchy(sheet, row):
    """
    Get cell description from columns A, B, C without bold header enrichment
    """
    print(f"DEBUG: Getting description for row {row}")
    
    # Check columns A, B, C in order to find the first non-empty cell
    for col_letter in ['A', 'B', 'C']:
        cell = sheet[f"{col_letter}{row}"]
        print(f"DEBUG: Checking {col_letter}{row} = {cell.value}")
        
        if cell.value is not None:
            cell_value = cell.value
            
            # If cell has a formula, follow it
            if cell.data_type == "f" or (isinstance(cell_value, str) and cell_value.startswith("=")):
                print(f"DEBUG: Found formula in {col_letter}{row}: {cell_value}")
                m = re.match(r"=([^!]+)!([A-Z]+[0-9]+)", cell_value)
                if m:
                    target_sheet, target_cell = m.groups()
                    try:
                        cell_value = wb_values[target_sheet][target_cell].value
                        print(f"DEBUG: Formula resolved to: {cell_value}")
                        if cell_value is None:
                            cell_value = ""
                        else:
                            cell_value = str(cell_value).strip()
                    except:
                        cell_value = ""
            
            result = str(cell_value).strip() if cell_value else ""
            print(f"DEBUG: Final result for row {row}: '{result}'")
            return result
    
    # If no cell found, return row number
    print(f"DEBUG: No cell found for row {row}, returning Row {row}")
    return f"Row {row}"

In [22]:
def build_column_period_map(wb_values, sheet_name, start_row=1, today_date=None, max_row=150):
    """
    Build column to period map (automated):
    - for each column, look up from start_row to max_row
    - first period string or number becomes the period
    - same name as before, so later code works
    """
    import datetime
    import re
    from openpyxl.utils import get_column_letter

    if today_date is None:
        today_date = datetime.date.today()

    sheet = wb_values[sheet_name]  # Access worksheet by name from Workbook
    col_to_period = {}

    # Enhanced pattern to match more period formats - same as lookup_column_period
    pattern = re.compile(r'\b(\d{1,2}Q\d{2,4}E?|\d{4}E?|FY\d{2,4}E?)\b', re.I)

    for col in range(1, sheet.max_column + 1):
        col_letter = get_column_letter(col)
        period = ""

        for row in range(start_row, min(max_row + 1, sheet.max_row + 1)):
            cell_obj = sheet[f"{col_letter}{row}"]
            cell_value = cell_obj.value

            if cell_value is None:
                continue

            # Process string values - look for period patterns
            if isinstance(cell_value, str):
                text = cell_value.strip()
                
                # Search for period pattern in the text
                match = pattern.search(text)
                if match:
                    period = match.group(1)  # Return the matched period
                    break

            # Process numeric values - handle years stored as numbers
            elif isinstance(cell_value, (int, float)):
                # Check if it's a 4-digit year
                if 2000 <= cell_value <= 2050:
                    year = int(cell_value)
                    # Determine if it's forecast based on year threshold
                    forecast_threshold = 2025  # Years >= 2025 are considered forecast
                    if year >= forecast_threshold:
                        period = f"{year}E"
                    else:
                        period = str(year)
                    break

            # Skip other types (dates, etc.)

        if period:
            col_to_period[col_letter] = period

    return col_to_period


def build_all_sheets_col_to_period(wb_values, today_date=None, max_row=150):
    """
    Build col_to_period maps for all sheets in the workbook
    """
    all_col_to_period = {}
    
    for sheet_name in wb_values.sheetnames:  # Use .sheetnames for openpyxl Workbook
        try:
            col_to_period = build_column_period_map(
                wb_values, sheet_name, start_row=1, today_date=today_date, max_row=max_row
            )
            if col_to_period:  # Only store if we found periods
                all_col_to_period[sheet_name] = col_to_period
                print(f"✓ {sheet_name}: Found {len(col_to_period)} period columns")
            else:
                print(f"⚠ {sheet_name}: No periods found")
        except Exception as e:
            print(f"✗ {sheet_name}: Error - {e}")
    
    return all_col_to_period


def lookup_column_period_from_map(all_col_to_period, sheet_name, cell_ref):
    """
    Helper function to get period for a specific cell from the all_col_to_period map
    """
    if sheet_name not in all_col_to_period:
        return ""
    
    col_letter = re.match(r"[A-Z]+", cell_ref).group(0)
    return all_col_to_period[sheet_name].get(col_letter, "")


# Usage example:
import datetime

manual_today_date = datetime.date(2025, 6, 24)

# Build col_to_period maps for all sheets
all_col_to_period = build_all_sheets_col_to_period(
    wb_values, 
    today_date=manual_today_date,
    max_row=150
)

# Test specific sheets
print("\n=== COLUMN TO PERIOD MAPPING RESULTS ===")
for sheet_name, col_to_period in all_col_to_period.items():
    print(f"\n{sheet_name}:")
    for col, period in sorted(col_to_period.items()):
        print(f"  {col}: {period}")

# Test the helper function
print(f"\nTest: Moon Model!PL45 period = '{lookup_column_period_from_map(all_col_to_period, 'Moon Model', 'CP45')}'")

✓ Summary: Found 9 period columns
✓ R$: Found 110 period columns
✓ ConsFin: Found 111 period columns
✓ Fundamental: Found 5 period columns
✓ Desc: Found 1 period columns
✓ OnePager: Found 42 period columns
✓ Revenue: Found 178 period columns
✓ Assumptions: Found 178 period columns
✓ Schedules: Found 180 period columns
✓ Moon Model: Found 130 period columns
✓ _IntelliSense_: Found 1 period columns
✓ Model: Found 178 period columns
✓ Ratio Analysis: Found 178 period columns
✓ SOP: Found 15 period columns
✓ DCF: Found 23 period columns
✓ CFROI: Found 34 period columns
✓ ROIC: Found 16 period columns
✓ ErrorChk: Found 78 period columns
✓ LinkC: Found 188 period columns
✓ LinkD: Found 178 period columns
✓ Link: Found 35 period columns
✓ Street Est: Found 35 period columns
⚠ Price Data: No periods found
⚠ Version: No periods found

=== COLUMN TO PERIOD MAPPING RESULTS ===

Summary:
  G: 2017
  H: 2018
  I: 2019
  J: 2020
  K: 2021E
  L: 2022E
  M: 2023E
  N: 2024E
  O: 2025E

R$:
  AA: 2005


In [23]:
import re
from openpyxl.utils.cell import get_column_letter
from openpyxl.utils import range_boundaries

def parse_sheet_reference(ref):
    """
    Parse a sheet reference like 'Sheet Name'!A1 or Sheet!A1
    Returns (sheet_name, cell_ref)
    """
    if "!" not in ref:
        return None, ref
    
    # Split by the last occurrence of ! to handle cases like 'Sheet!Name'!A1
    parts = ref.rsplit("!", 1)
    sheet_part = parts[0]
    cell_part = parts[1]
    
    # Remove quotes if present
    if sheet_part.startswith("'") and sheet_part.endswith("'"):
        sheet_part = sheet_part[1:-1]
    
    return sheet_part, cell_part

In [24]:

def trace_merged(wb_values, wb_formulas, sheet_name, cell_ref, all_col_to_period, max_depth=20, visited=None, depth=0, path=None):
    def is_terminal_cell(sheet_name, cell_ref, formula):
        col_letter = re.match(r"[A-Z]+", cell_ref).group(0)
        # Updated to use the new lookup function
        period = lookup_column_period_from_map(all_col_to_period, sheet_name, cell_ref)
        if not period:
            return True
        if sheet_name.lower().startswith("schedule"):
            return True
        if formula is None:
            return True
        if isinstance(formula, str) and "[" in formula:
            return True
        if re.match(r"^=[0-9.\-+*/ ()]+$", formula):
            return True
        return False

    if visited is None:
        visited = set()
    if path is None:
        path = []

    key = (sheet_name, cell_ref)
    if key in visited or depth > max_depth:
        return []

    visited.add(key)
    current_path = path + [f"{sheet_name}!{cell_ref}"]

    sheet_v = wb_values[sheet_name]
    sheet_f = wb_formulas[sheet_name]
    cell_v = sheet_v[cell_ref]

    try:
        cell_f = sheet_f[cell_ref]
        formula = cell_f.value if isinstance(cell_f.value, str) and cell_f.value.startswith("=") else None
        description = resolve_label_with_hierarchy(sheet_f, cell_f.row) or f"Row {cell_f.row}"
    except KeyError:
        formula = None
        description = None

    value = cell_v.value
    external_ref_segment = ""
    is_external = False

    if formula and "[" in formula:
        match_ref = re.search(r"(\[[^\]]+\][^\+\-\*/\)^\n]*)", formula)
        external_ref_segment = match_ref.group(1) if match_ref else "[EXTERNAL]"
        is_external = True

    is_terminal = is_terminal_cell(sheet_name, cell_ref, formula)

    numeric_literals = []
    if not is_external and formula:
        # Remove quoted strings first
        cleaned_formula = re.sub(r'"[^"]*"', '', formula)
        
        # Find all cell references (including those with $ and sheet names)
        cell_refs = re.findall(r'(?<![0-9A-Za-z_])(?:\'[^\']*\'!\$?[A-Z]{1,3}\$?[0-9]{1,5}|[A-Za-z0-9_]+!\$?[A-Z]{1,3}\$?[0-9]{1,5}|\$?[A-Z]{1,3}\$?[0-9]{1,5})(?![0-9A-Za-z_])', cleaned_formula)
        
        # Remove all cell references from the formula to isolate numeric constants
        temp_formula = cleaned_formula
        for ref in cell_refs:
            temp_formula = temp_formula.replace(ref, '')
        
        # Extract numeric constants
        all_numbers = re.findall(r'[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?', temp_formula)
        filtered_constants = []
        for num_str in all_numbers:
            try:
                val = float(num_str)
                if val == 1.0 and (re.search(r'\(1\s*\+\s*[\w\d]', formula) or re.search(r'[\w\d]\s*\+\s*1\)', formula)):
                    continue
                if val == -1.0 and formula.strip().endswith("-1"):
                    continue
                filtered_constants.append(val)
            except:
                continue
        numeric_literals = filtered_constants

    # Updated period lookup - use the new function
    period = lookup_column_period_from_map(all_col_to_period, sheet_name, cell_ref)

    output = [{
        "sheet": sheet_name,
        "cell": cell_ref,
        "formula": formula,
        "value": value,
        "depth": depth,
        "description": description,
        "is_terminal": is_terminal or is_external,
        "external_ref_segment": external_ref_segment,
        "constants": numeric_literals,
        "path": current_path,
        "period": period
    }]

    if is_terminal or is_external:
        return output

    refs = []
    ranges = []
    if isinstance(formula, str):
        # Updated regex to handle quoted sheet names AND $ symbols for absolute references
        # Pattern explanation:
        # - (?:'[^']*'!\$?[A-Z]{1,3}\$?[0-9]{1,5}) - quoted sheet with absolute refs
        # - (?:[A-Za-z0-9_]+!\$?[A-Z]{1,3}\$?[0-9]{1,5}) - unquoted sheet with absolute refs  
        # - (?:\$?[A-Z]{1,3}\$?[0-9]{1,5}) - local cell with absolute refs
        refs = re.findall(r"(?<![0-9A-Za-z_])(?:'[^']*'!\$?[A-Z]{1,3}\$?[0-9]{1,5}|[A-Za-z0-9_]+!\$?[A-Z]{1,3}\$?[0-9]{1,5}|\$?[A-Z]{1,3}\$?[0-9]{1,5})(?![0-9A-Za-z_])", formula)
        refs = [ref.replace("$", "") for ref in refs]
        
        # Updated regex for ranges with quoted sheet names AND $ symbols
        ranges = re.findall(r"('[^']*'!\$?[A-Z]{1,3}\$?[0-9]{1,5}:\$?[A-Z]{1,3}\$?[0-9]{1,5}|[A-Za-z0-9_]+!\$?[A-Z]{1,3}\$?[0-9]{1,5}:\$?[A-Z]{1,3}\$?[0-9]{1,5}|\$?[A-Z]{1,3}\$?[0-9]{1,5}:\$?[A-Z]{1,3}\$?[0-9]{1,5})", formula)
        ranges = [rng.replace("$", "") for rng in ranges]

    for ref in refs:
        # Use the new parsing function
        other_sheet, other_cell = parse_sheet_reference(ref)
        if other_sheet is None:
            other_sheet = sheet_name
            other_cell = ref

        col_letter = re.match(r"[A-Z]+", other_cell).group(0)

        # Updated filtering logic
        if other_sheet.lower().startswith("schedule"):
            continue
        
        # Check if this sheet has any period mapping at all
        if other_sheet not in all_col_to_period:
            continue
            
        # Check if this specific column has a period
        if col_letter not in all_col_to_period[other_sheet]:
            continue

        # Check if the sheet exists before trying to trace
        if other_sheet not in wb_values:
            print(f"Warning: Sheet '{other_sheet}' not found in workbook")
            continue

        output += trace_merged(wb_values, wb_formulas, other_sheet, other_cell, all_col_to_period, max_depth, visited, depth + 1, current_path)

    for rng in ranges:
        # Use the new parsing function for ranges too
        other_sheet, cell_range = parse_sheet_reference(rng)
        if other_sheet is None:
            other_sheet = sheet_name
            cell_range = rng

        min_col, min_row, max_col, max_row = range_boundaries(cell_range)
        for row in range(min_row, max_row + 1):
            for col in range(min_col, max_col + 1):
                col_letter = get_column_letter(col)
                
                # Updated filtering logic for ranges
                if other_sheet.lower().startswith("schedule"):
                    continue
                
                # Check if this sheet has any period mapping at all
                if other_sheet not in all_col_to_period:
                    continue
                    
                # Check if this specific column has a period
                if col_letter not in all_col_to_period[other_sheet]:
                    continue
                
                # Check if the sheet exists before trying to trace
                if other_sheet not in wb_values:
                    print(f"Warning: Sheet '{other_sheet}' not found in workbook")
                    continue
                    
                cell = f"{col_letter}{row}"
                output += trace_merged(wb_values, wb_formulas, other_sheet, cell, all_col_to_period, max_depth, visited, depth + 1, current_path)

    return output


# Updated usage
trace = trace_merged(wb_values, wb_formulas, TARGET_SHEET, TARGET_CELL, all_col_to_period, max_depth=30)

for item in trace:
    sheet = item['sheet']
    cell = item['cell']
    cell_loc = f"{sheet}!{cell}"

    # Extract value directly
    cell_value = wb_values[sheet][cell].value
    if isinstance(cell_value, str):
        name = cell_value.strip()
    else:
        name = item.get('description') or ''

    formula = item.get('formula') or ''
    value = item.get('value')
    depth = item.get('depth')
    external_ref = item.get("external_ref_segment", "")

    col = re.match(r"[A-Z]+", cell).group(0)
    period = item["period"]

    label = f"{name} ({cell_loc})" if name else cell_loc
    value_str = f"{value:.4f}" if isinstance(value, (int, float)) else (value or "")
    constants = item.get("constants", [])
    const_str = ", ".join(f"{c:.4f}" for c in constants) if constants else ""

    print(
        f"{label:<50} | Period: {period:<6} | Formula: {formula:<50} | "
        f"Value: {value_str:<12} | Depth: {depth} | External Ref: {external_ref} | Constants: {const_str}"
    )

DEBUG: Getting description for row 137
DEBUG: Checking A137 = EPS (Fully diluted)
DEBUG: Final result for row 137: 'EPS (Fully diluted)'
DEBUG: Getting description for row 133
DEBUG: Checking A133 = Shares Outstanding (mm) (Fully Diluted)
DEBUG: Final result for row 133: 'Shares Outstanding (mm) (Fully Diluted)'
DEBUG: Getting description for row 133
DEBUG: Checking A133 = Shares Outstanding (mm) (Fully Diluted)
DEBUG: Final result for row 133: 'Shares Outstanding (mm) (Fully Diluted)'
DEBUG: Getting description for row 133
DEBUG: Checking A133 = Shares Outstanding (mm) (Fully Diluted)
DEBUG: Final result for row 133: 'Shares Outstanding (mm) (Fully Diluted)'
DEBUG: Getting description for row 133
DEBUG: Checking A133 = Shares Outstanding (mm) (Fully Diluted)
DEBUG: Final result for row 133: 'Shares Outstanding (mm) (Fully Diluted)'
DEBUG: Getting description for row 133
DEBUG: Checking A133 = Shares Outstanding (mm) (Fully Diluted)
DEBUG: Final result for row 133: 'Shares Outstanding 

In [25]:
# --- Filter trace for E-period, non-zero ---
# Improved version with better error handling and floating-point precision

def is_effectively_zero(value, tolerance=1e-10):
    """Check if a value is effectively zero, accounting for floating-point precision"""
    if value is None:
        return True
    if isinstance(value, (int, float)):
        return abs(value) < tolerance
    return False

def is_error_value(value):
    """Check if a value is an Excel error (starts with #)"""
    if value is None:
        return False
    return str(value).startswith("#")

filtered_trace = [
    item for item in trace
    if item["period"].endswith("E")
    and not is_effectively_zero(item.get("value"))
    and not is_error_value(item.get("value"))
]
'''
# Alternative: Your original code with minor improvements
filtered_trace_original = [
    item for item in trace
    if item["period"].endswith("E")
    and item.get("value") not in (0, 0.0, None)
    and not str(item.get("value", "")).startswith("#")  # Added default empty string
]
'''
# --- Print filtered ---
print("\n--- Filtered Forecast E Cells ---\n")

for item in filtered_trace:
    period = item["period"]
    sheet = item['sheet']
    cell = item['cell']
    label = f"{item.get('description', '')} ({sheet}!{cell})"
    value = item.get('value')
    depth = item.get('depth')
    formula = item.get('formula') or ''
    constants = item.get("constants", [])
    const_str = ", ".join(f"{c:.4f}" for c in constants) if constants else ""
    external_ref = item.get("external_ref_segment", "")
    value_str = f"{value:.4f}" if isinstance(value, (float, int)) else str(value)

    print(
        f"{label:<50} | Period: {period:<6} | Formula: {formula:<50} | "
        f"Value: {value_str:<12} | Depth: {depth} | External Ref: {external_ref} | Constants: {const_str}"
    )


--- Filtered Forecast E Cells ---

EPS (Fully diluted) (Model!ET137)                  | Period: 2025E  | Formula: =IF(ISNUMBER(1/ET133),((ET114-ET135)/(ET133/Fundamental!$C$54)*Fundamental!$E$58)/ET$514,0) | Value: 1.2177       | Depth: 0 | External Ref:  | Constants: 1.0000, 0.0000
Shares Outstanding (mm) (Fully Diluted) (Model!ET133) | Period: 2025E  | Formula: =+ES133                                            | Value: 948.3754     | Depth: 1 | External Ref:  | Constants: 
Shares Outstanding (mm) (Fully Diluted) (Model!ES133) | Period: 4Q25E  | Formula: =ER133                                             | Value: 948.3754     | Depth: 2 | External Ref:  | Constants: 
Shares Outstanding (mm) (Fully Diluted) (Model!ER133) | Period: 3Q25E  | Formula: =EQ133                                             | Value: 948.3754     | Depth: 3 | External Ref:  | Constants: 
Shares Outstanding (mm) (Fully Diluted) (Model!EQ133) | Period: 2Q25E  | Formula: =EP133                                    

In [26]:
# --- Enhanced Terminal input detector (FIXED) ---
def is_terminal_input(item):
    period = item["period"]
    if not period.endswith("E"):
        return False

    formula = item.get("formula")
    constants = item.get("constants", [])
    
    # Check for meaningful constants
    has_meaningful_constants = False
    if constants:
        # Exclude common time-based/calculation constants
        excluded_constants = {0.25, 0.33, 0.5, 0.67, 0.75, 1/4, 1/3, 1/2, 2/3, 3/4, 
                            1, -1, 2, 3, 4, 52, 90, 100, 
                            180, 270, 365, 1000, 10000, 100000, 1000000}
        meaningful_constants = [c for c in constants if c not in excluded_constants]
        if meaningful_constants:
            has_meaningful_constants = True
    
    # Original logic: no formula or external references
    if not formula or "[" in formula:
        return True
    
    # If formula only contains constants (no cell references), it's terminal
    if re.match(r"^=[0-9.\-+*/ ()]+$", formula):
        return True

    # NEW: If we have meaningful constants, include it regardless of references
    # This handles cases like "=+ET139-0.02" where -0.02 is a meaningful manual input
    if has_meaningful_constants:
        # Exclude complex formulas where constants are likely technical
        complex_keywords = ["IF", "CONCATENATE", "VLOOKUP", "HLOOKUP", "INDEX", "MATCH", 
                          "CHOOSE", "OFFSET", "INDIRECT", "SUMIF", "COUNTIF", "AVERAGEIF"]
        formula_upper = formula.upper()
        
        for keyword in complex_keywords:
            if keyword in formula_upper:
                return False  # Exclude complex formulas
        
        return True  # Include simple formulas with meaningful constants

    # Extract all cell references from the formula
    refs = re.findall(
        r'(?<![0-9A-Za-z_])(?:\'[^\']*\'!\$?[A-Z]{1,3}\$?[0-9]{1,5}|[A-Za-z0-9_]+!\$?[A-Z]{1,3}\$?[0-9]{1,5}|\$?[A-Z]{1,3}\$?[0-9]{1,5})(?![0-9A-Za-z_])',
        formula
    )

    # Check each referenced cell
    for ref in refs:
        ref = ref.replace("$", "")
        
        # Parse the sheet reference properly
        ref_sheet, ref_cell = parse_sheet_reference(ref)
        if ref_sheet is None:
            ref_sheet = item['sheet']  # Same sheet reference

        # First, try to find the referenced cell in our trace
        ref_found_in_trace = False
        for ref_item in trace:
            if ref_item["sheet"] == ref_sheet and ref_item["cell"] == ref_cell:
                ref_found_in_trace = True
                ref_period = ref_item["period"]
                # If any referenced cell is also a forecast (ends with E), this is NOT terminal
                if ref_period.endswith("E"):
                    return False
                break
        
        # If not found in trace, check using the all_col_to_period mapping
        if not ref_found_in_trace:
            ref_period = lookup_column_period_from_map(all_col_to_period, ref_sheet, ref_cell)
            # If the referenced cell is in a forecast period, this is NOT terminal
            if ref_period.endswith("E"):
                return False

    # If we get here, all referenced cells are either historical (non-E) or external
    return True


# --- Select terminal inputs ---
terminal_inputs = [
    item for item in filtered_trace
    if is_terminal_input(item)
    and isinstance(item.get("value"), (int, float))
    and item.get("value") not in (0, 0.0)
]

# --- Print terminal inputs with inclusion reasoning ---
print("\n--- Enhanced Terminal Input Cells (FIXED) ---\n")

included_count = 0
excluded_count = 0

for item in filtered_trace:
    if not item["period"].endswith("E"):
        continue
    if not isinstance(item.get("value"), (int, float)):
        continue
    if item.get("value") in (0, 0.0):
        continue
    
    period = item["period"]
    sheet = item['sheet']
    cell = item['cell']
    label = f"{item.get('description', '')} ({sheet}!{cell})"
    value = item.get('value')
    depth = item.get('depth')
    formula = item.get('formula') or ''
    constants = item.get("constants", [])
    const_str = ", ".join(f"{c:.4f}" for c in constants) if constants else ""
    external_ref = item.get("external_ref_segment", "")
    value_str = f"{value:.4f}" if isinstance(value, (float, int)) else str(value)
    
    # Determine inclusion status and reason
    is_included = is_terminal_input(item)
    
    if is_included:
        status = "✅ INCLUDED"
        included_count += 1
        
        # Determine reason for inclusion
        if not formula or "[" in formula:
            reason = "No formula/External ref"
        elif re.match(r"^=[0-9.\-+*/ ()]+$", formula):
            reason = "Constants only"
        elif constants:
            excluded_constants = {0.25, 0.33, 0.5, 0.67, 0.75, 1/4, 1/3, 1/2, 2/3, 3/4, 
                                1, -1, 2, 3, 4, 52, 90, 100, 
                                180, 270, 365, 1000, 10000, 100000, 1000000}
            meaningful_constants = [c for c in constants if c not in excluded_constants]
            if meaningful_constants:
                reason = f"Meaningful constants: {meaningful_constants}"
            else:
                reason = "Historical refs only"
        else:
            reason = "Historical refs only"
    else:
        status = "❌ EXCLUDED"
        excluded_count += 1
        
        # Determine reason for exclusion
        if constants:
            excluded_constants = {0.25, 0.33, 0.5, 0.67, 0.75, 1/4, 1/3, 1/2, 2/3, 3/4, 
                                1, -1, 2, 3, 4, 52, 90, 100, 
                                180, 270, 365, 1000, 10000, 100000, 1000000}
            meaningful_constants = [c for c in constants if c not in excluded_constants]
            if meaningful_constants:
                complex_keywords = ["IF", "CONCATENATE", "VLOOKUP", "HLOOKUP", "INDEX", "MATCH", 
                              "CHOOSE", "OFFSET", "INDIRECT", "SUMIF", "COUNTIF", "AVERAGEIF"]
                formula_upper = formula.upper()
                for keyword in complex_keywords:
                    if keyword in formula_upper:
                        reason = f"Complex formula ({keyword})"
                        break
                else:
                    reason = "References forecast periods"
            else:
                reason = "References forecast periods"
        else:
            reason = "References forecast periods"
    
    print(
        f"{status} | {label:<50} | Period: {period:<6} | Formula: {formula:<50} | "
        f"Value: {value_str:<12} | Constants: {const_str:<20} | Reason: {reason}"
    )

print(f"\n📊 Summary: {included_count} included, {excluded_count} excluded")
print(f"🔍 Review the above list to ensure business-meaningful constants are captured correctly!")


--- Enhanced Terminal Input Cells (FIXED) ---

❌ EXCLUDED | EPS (Fully diluted) (Model!ET137)                  | Period: 2025E  | Formula: =IF(ISNUMBER(1/ET133),((ET114-ET135)/(ET133/Fundamental!$C$54)*Fundamental!$E$58)/ET$514,0) | Value: 1.2177       | Constants: 1.0000, 0.0000       | Reason: Complex formula (IF)
❌ EXCLUDED | Shares Outstanding (mm) (Fully Diluted) (Model!ET133) | Period: 2025E  | Formula: =+ES133                                            | Value: 948.3754     | Constants:                      | Reason: References forecast periods
❌ EXCLUDED | Shares Outstanding (mm) (Fully Diluted) (Model!ES133) | Period: 4Q25E  | Formula: =ER133                                             | Value: 948.3754     | Constants:                      | Reason: References forecast periods
❌ EXCLUDED | Shares Outstanding (mm) (Fully Diluted) (Model!ER133) | Period: 3Q25E  | Formula: =EQ133                                             | Value: 948.3754     | Constants:                     

In [27]:
import re
from collections import defaultdict

def get_terminal_cell_path(terminal_item):
    return f"{terminal_item['sheet']}!{terminal_item['cell']}"

def find_terminal_chain(filtered_trace, terminal_item):
    """
    Find the full path (chain of cell references) for a given terminal item.
    """
    terminal_path = get_terminal_cell_path(terminal_item)
    for item in filtered_trace:
        if get_terminal_cell_path(item) == terminal_path:
            return item["path"]
    return []

def filter_chain_by_period(chain_paths, terminal_item, all_col_to_period):
    """
    Keep only items in the same period as the terminal.
    Returns list of (sheet, cell) tuples AND the filtered path strings for printing.
    """
    terminal_period = terminal_item["period"]
    filtered_tuples = []
    filtered_paths = []
    
    for path in chain_paths:
        sheet, cell = path.split("!")
        period = lookup_column_period_from_map(all_col_to_period, sheet, cell)
        if period == terminal_period:
            filtered_tuples.append((sheet, cell))
            filtered_paths.append(path)
    
    return filtered_tuples, filtered_paths

def get_trace_lookup(trace):
    return {f"{item['sheet']}!{item['cell']}": item for item in trace}

def get_nearest_parents_descriptions(filtered_chain, terminal_item, trace_lookup, num_parents=2):
    """
    From filtered chain, return up to two parent descriptions above the terminal cell.
    Also tries to find adjacent row descriptions if chain is too short.
    """
    terminal_path = get_terminal_cell_path(terminal_item)
    terminal_row = int(re.match(r"[A-Z]+(\d+)", terminal_item["cell"]).group(1))
    terminal_sheet = terminal_item["sheet"]
    terminal_col = re.match(r"([A-Z]+)\d+", terminal_item["cell"]).group(1)

    descriptions = []
    
    # First, try to get descriptions from the filtered chain
    for sheet, cell in reversed(filtered_chain):
        path = f"{sheet}!{cell}"
        if path == terminal_path:
            continue
        row = int(re.match(r"[A-Z]+(\d+)", cell).group(1))
        if sheet == terminal_sheet and row == terminal_row:
            continue
        desc = trace_lookup.get(path, {}).get("description", "")
        if desc and desc not in descriptions:
            descriptions.append(desc)
        if len(descriptions) == num_parents:
            break
    
    # If we don't have enough descriptions from chain, try adjacent rows
    if len(descriptions) < num_parents and len(filtered_chain) <= 1:
        print(f"DEBUG PARENTS: Chain too short ({len(filtered_chain)}), checking adjacent rows for {terminal_path}")
        
        # Check rows above the terminal (row-1, row-2, etc.)
        for offset in range(1, 5):  # Check up to 4 rows above
            if len(descriptions) >= num_parents:
                break
            adjacent_row = terminal_row - offset
            if adjacent_row > 0:
                adjacent_path = f"{terminal_sheet}!{terminal_col}{adjacent_row}"
                desc = trace_lookup.get(adjacent_path, {}).get("description", "")
                if desc and desc not in descriptions and desc != terminal_item.get("description", ""):
                    descriptions.append(desc)
                    print(f"DEBUG PARENTS: Found adjacent row description: {adjacent_path} -> '{desc}'")
    
    return list(reversed(descriptions))  # Return in order: parent1, parent2

def get_enriched_terminal_description(terminal_item, trace, all_col_to_period):
    """
    Enhanced version that returns both enriched description and chain info for printing.
    Format: branch → subbranch → terminal_description
    """
    trace_lookup = get_trace_lookup(trace)
    terminal_path = get_terminal_cell_path(terminal_item)

    full_chain = find_terminal_chain(trace, terminal_item)
    if not full_chain:
        return terminal_item.get("description", ""), None, None

    # Get both filtered tuples and paths for printing
    filtered_chain, filtered_paths = filter_chain_by_period(full_chain, terminal_item, all_col_to_period)
    
    if not filtered_chain:
        return terminal_item.get("description", ""), None, None

    parents = get_nearest_parents_descriptions(filtered_chain, terminal_item, trace_lookup)
    
    # Build enriched description: parents + terminal description (terminal at the end)
    terminal_desc = terminal_item.get("description", "")
    enriched_parts = parents + [terminal_desc] if terminal_desc else parents
    enriched_desc = " → ".join(p for p in enriched_parts if p)
    
    # If no enrichment occurred (no parents found), just return original description
    if not parents:
        enriched_desc = terminal_desc
    
    # Return enriched description, full chain info, and filtered chain info
    chain_info = {
        "full_chain": full_chain,
        "filtered_chain": filtered_paths,
        "full_length": len(full_chain),
        "filtered_length": len(filtered_paths)
    }
    
    return enriched_desc, chain_info, parents

def run_enrichment_for_all_terminals(terminal_inputs, trace, all_col_to_period):
    """
    Enrich and print descriptions for all terminal input cells with chain printing.
    """
    enriched_results = []

    print("\n🔍 Enriching Terminal Inputs (Period-Matched Nearest Parents)\n")

    for i, item in enumerate(terminal_inputs, 1):
        print(f"\n[{i}/{len(terminal_inputs)}] Processing: {item['sheet']}!{item['cell']}")
        print(f"Original description: '{item.get('description', '')}' | Period: {item['period']} | Value: {item.get('value')}")

        enriched_desc, chain_info, parents = get_enriched_terminal_description(item, trace, all_col_to_period)
        
        # Print chain information like in Image 1
        if chain_info:
            print(f"DEBUG CHAIN: Full chain length {chain_info['full_length']}: {'  →  '.join(chain_info['full_chain'])}")
            print(f"DEBUG CHAIN: Filtered chain length {chain_info['filtered_length']} (same period only): {'  →  '.join(chain_info['filtered_chain'])}")
            if parents:
                print(f"DEBUG CHAIN: Selected parents: {parents}")
        else:
            print("DEBUG CHAIN: No chain found")
        
        enriched_results.append({
            "sheet": item["sheet"],
            "cell": item["cell"],
            "period": item["period"],
            "value": item.get("value"),
            "depth": item.get("depth"),
            "original_desc": item.get("description", ""),
            "enriched_desc": enriched_desc,
            "chain_info": chain_info
        })

        print(f"✅ Enriched: '{enriched_desc}'")

    # Summary
    print("\n" + "="*100)
    print("SUMMARY: Terminal Cell Enrichment (Nearest Parents in Same Period)")
    print("="*100)

    enriched_count = 0
    for result in enriched_results:
        enriched = result["enriched_desc"]
        original = result["original_desc"]
        enriched_flag = "✅ ENRICHED" if ("→" in enriched and enriched != original) else "❌ NO ENRICHMENT"
        if enriched_flag == "✅ ENRICHED":
            enriched_count += 1

        # Show chain length in summary
        chain_length = result["chain_info"]["filtered_length"] if result["chain_info"] else 0
        
        # Only show the final enriched description (no duplication)
        print(f"{enriched_flag} | {result['sheet']}!{result['cell']:<10} | {result['period']} | Depth: {result['depth']} | Chain: {chain_length} | '{enriched}'")

    total = len(enriched_results)
    print(f"\n📊 {enriched_count}/{total} enriched ({(enriched_count/total)*100:.1f}%)")

    return enriched_results

# Usage (assuming you have the required variables):
enriched_results = run_enrichment_for_all_terminals(terminal_inputs, filtered_trace, all_col_to_period)


🔍 Enriching Terminal Inputs (Period-Matched Nearest Parents)


[1/49] Processing: Model!EQ133
Original description: 'Shares Outstanding (mm) (Fully Diluted)' | Period: 2Q25E | Value: 948.3754128773256
DEBUG PARENTS: Chain too short (1), checking adjacent rows for Model!EQ133
DEBUG CHAIN: Full chain length 5: Model!ET137  →  Model!ET133  →  Model!ES133  →  Model!ER133  →  Model!EQ133
DEBUG CHAIN: Filtered chain length 1 (same period only): Model!EQ133
✅ Enriched: 'Shares Outstanding (mm) (Fully Diluted)'

[2/49] Processing: Moon Model!CL8
Original description: 'yoy' | Period: 2Q25E | Value: 0.02
DEBUG PARENTS: Chain too short (1), checking adjacent rows for Moon Model!CL8
DEBUG PARENTS: Found adjacent row description: Moon Model!CL7 -> '% YoY'
DEBUG PARENTS: Found adjacent row description: Moon Model!CL6 -> 'Net Retail Revenue'
DEBUG CHAIN: Full chain length 15: Model!ET137  →  Model!ET114  →  Model!ET110  →  Model!ET98  →  Model!ET49  →  Model!ET28  →  Model!ET9  →  Revenue!ET56  →  R

In [28]:
import datetime
from openpyxl.utils import column_index_from_string, get_column_letter

def should_force_zero(value, description, number_format):
    """
    Returns True if the value and description indicate a growth/change input that should be forced to 0.
    """
    if not isinstance(value, (int, float)):
        return False

    if not description:
        return False

    desc_lower = description.lower()

    # Keywords for growth/change situationsa
    keywords = [
        "% change", "growth", "yoy", "yr/yr", "q/q", "mom", "y/y", "m/m",
        "variance", "var", "delta", "change", "chg", "chng",
        "inflation", "cpi", "ppi", "fx impact", "discount rate change",
        "wacc delta", "wacc change", "risk premium change",
        "margin delta", "cost increase rate", "tax change", "tax rate change"
    ]

    for kw in keywords:
        if kw in desc_lower:
            if "margin %" in desc_lower or "tax rate %" in desc_lower:
                return False
            return True

    return False


def parse_sheet_reference(ref):
    """
    Parse a sheet reference like 'Sheet Name'!A1 or Sheet!A1
    Returns (sheet_name, cell_ref)
    """
    if "!" not in ref:
        return None, ref
    
    # Split by the last occurrence of ! to handle cases like 'Sheet!Name'!A1
    parts = ref.rsplit("!", 1)
    sheet_part = parts[0]
    cell_part = parts[1]
    
    # Remove quotes if present
    if sheet_part.startswith("'") and sheet_part.endswith("'"):
        sheet_part = sheet_part[1:-1]
    
    return sheet_part, cell_part

# Create lookup dictionary from enriched results (from previous step)
# This assumes you have 'enriched_results' from your test function
enriched_descriptions = {}
for result in enriched_results:
    cell_key = f"{result['sheet']}!{result['cell']}"
    enriched_descriptions[cell_key] = result['enriched_desc']

print(f"📊 Created enriched description lookup for {len(enriched_descriptions)} cells")

# Create mapping for all sheets with terminal cells
sheet_names = {item['sheet'] for item in terminal_inputs}

# Use a date that's earlier than your forecast periods to ensure they get "E" suffix
# Setting it to March 2025 so that Jun-25, Sep-25, etc. will all get "E"
manual_today_date = datetime.date(2025, 4, 1)

# Use the all_col_to_period mapping that was already built, but filter for relevant sheets
sheet_to_col_period = {}
for sheet in sheet_names:
    if sheet in all_col_to_period:
        sheet_to_col_period[sheet] = all_col_to_period[sheet]
    else:
        # If the sheet wasn't in the original mapping, build it separately
        sheet_to_col_period[sheet] = build_column_period_map(
            wb_values, sheet, start_row=1, today_date=manual_today_date, max_row=150
        )
    
    # Debug: Print the updated mapping
    #print(f"📊 Debug: Updated column periods for {sheet}: {sheet_to_col_period[sheet]}")

# Create a mapping from terminal inputs to get the correct period format
terminal_period_map = {}
for item in terminal_inputs:
    sheet = item["sheet"]
    cell = item["cell"]
    period = item["period"]  # This is in the correct format like 2Q25E, 3Q25E
    col = re.match(r"[A-Z]+", cell).group(0)
    
    if sheet not in terminal_period_map:
        terminal_period_map[sheet] = {}
    terminal_period_map[sheet][col] = period

baseline_inputs = []
recorded_cells = set()  # avoid duplicates

for item in terminal_inputs:
    sheet = item["sheet"]
    cell = item["cell"]
    row = int(re.match(r"[A-Z]+(\d+)", cell).group(1))
    desc = item.get("description", "").lower()

    col_period_map = sheet_to_col_period[sheet]

    # Identify latest historical value in the row
    #print(f"📊 Debug: All column periods for {sheet}: {col_period_map}")
    hist_cols = [col for col, period in col_period_map.items() if not period.endswith("E")]
    #print(f"📊 Debug: Historical columns found: {hist_cols}")
    
    if not hist_cols:
        print(f"⚠️ No historical columns found in {sheet} — skipping baseline for row {row}.")
        continue
    
    # Find the first forecast column to establish the boundary
    forecast_cols = [col for col, period in col_period_map.items() if period.endswith("E")]
    if forecast_cols:
        # Sort forecast columns by position to find the earliest forecast column
        forecast_cols_sorted = sorted(forecast_cols, key=lambda col: column_index_from_string(col))
        first_forecast_col = forecast_cols_sorted[0]
        first_forecast_index = column_index_from_string(first_forecast_col)
        #print(f"📊 Debug: First forecast column: {first_forecast_col} (index {first_forecast_index})")
        
        # Filter historical columns to only include those before the first forecast column
        hist_cols_before_forecast = [col for col in hist_cols if column_index_from_string(col) < first_forecast_index]
        #print(f"📊 Debug: Historical columns before forecast: {hist_cols_before_forecast}")
        
        if hist_cols_before_forecast:
            hist_cols_to_use = hist_cols_before_forecast
        else:
            # Fallback to all historical columns if none found before forecast
            hist_cols_to_use = hist_cols
    else:
        # No forecast columns found, use all historical columns
        hist_cols_to_use = hist_cols
    
    # Check if this is a full-year forecast period that needs prior year reference
    terminal_period = item["period"]
    if terminal_period and re.match(r'^\d{4}E$', terminal_period):  # Full year like "2025E"
        # For full-year periods, find the prior full year
        year = int(terminal_period.replace('E', ''))
        prior_year = str(year - 1)
        
        # Find column with the prior year period
        prior_year_cols = [col for col, period in col_period_map.items() if period == prior_year]
        if prior_year_cols:
            latest_hist_col = prior_year_cols[0]  # Use the first match
        else:
            # Fallback to standard logic if prior year column not found
            hist_cols_sorted = sorted(hist_cols_to_use, key=lambda col: column_index_from_string(col))
            latest_hist_col = hist_cols_sorted[-1]
    else:
        # Standard logic for quarterly/half-year periods
        hist_cols_sorted = sorted(hist_cols_to_use, key=lambda col: column_index_from_string(col))
        latest_hist_col = hist_cols_sorted[-1]  # Get the last (rightmost) historical column

    latest_hist_cell = f"{latest_hist_col}{row}"
    
    try:
        latest_hist_val = wb_values[sheet][latest_hist_cell].value
        #print(f"📊 Debug: Reading historical value from {sheet}!{latest_hist_cell} = {latest_hist_val}")
    except Exception as e:
        print(f"⚠️ Error reading historical value from {sheet}!{latest_hist_cell}: {e}")
        latest_hist_val = None

    original_val = wb_values[sheet][cell].value
    #print(f"📊 Debug: Original value from {sheet}!{cell} = {original_val}")

    # Decide baseline value
    number_format = wb_values[sheet][cell].number_format
    if should_force_zero(original_val, item.get("description", ""), number_format):
        baseline_val, baseline_source = 0, "forced_zero_due_to_indicator"
        #print(f"📊 Debug: Forcing to zero due to growth indicator")
    else:
        formula = item.get("formula") or ""
        if formula:
            #print(f"📊 Debug: Processing formula: {formula}")
            # Use the improved regex pattern that handles quoted sheet names and $ symbols
            ref_cells = re.findall(
                r'(?<![0-9A-Za-z_])(?:\'[^\']*\'!\$?[A-Z]{1,3}\$?[0-9]{1,5}|[A-Za-z0-9_]+!\$?[A-Z]{1,3}\$?[0-9]{1,5}|\$?[A-Z]{1,3}\$?[0-9]{1,5})(?![0-9A-Za-z_])',
                formula
            )
            ref_cells = [ref.replace("$", "") for ref in ref_cells]

            if len(ref_cells) == 1:
                ref = ref_cells[0]
                # Use the improved sheet parsing function
                ref_sheet, ref_cell = parse_sheet_reference(ref)
                if ref_sheet is None:
                    ref_sheet = sheet

                try:
                    ref_row = int(re.match(r"[A-Z]+(\d+)", ref_cell).group(1))
                    ref_col = re.match(r"[A-Z]+", ref_cell).group(0)
                    
                    # Use the all_col_to_period mapping to check if the reference is forecasted
                    ref_period = lookup_column_period_from_map(all_col_to_period, ref_sheet, ref_cell)

                    if not ref_period.endswith("E"):
                        try:
                            baseline_val = wb_values[ref_sheet][ref_cell].value
                            baseline_source = f"{ref_sheet}!{ref_cell}"
                            #print(f"📊 Debug: Using reference value from {baseline_source} = {baseline_val}")
                        except Exception as e:
                            print(f"⚠️ Error reading reference value from {ref_sheet}!{ref_cell}: {e}")
                            baseline_val, baseline_source = latest_hist_val, latest_hist_cell
                    else:
                        baseline_val, baseline_source = latest_hist_val, latest_hist_cell
                        #print(f"📊 Debug: Reference is forecasted, using historical value from {baseline_source} = {baseline_val}")
                except Exception as e:
                    print(f"⚠️ Error parsing reference cell {ref}: {e}")
                    baseline_val, baseline_source = latest_hist_val, latest_hist_cell
            else:
                # Multiple refs or no clear precedent → fallback to latest historical period same row
                baseline_val, baseline_source = latest_hist_val, latest_hist_cell
                #print(f"📊 Debug: Multiple/no refs, using historical value from {baseline_source} = {baseline_val}")
        else:
            # NO FORMULA - this is a hardcoded value, should use historical value
            baseline_val, baseline_source = latest_hist_val, latest_hist_cell
            #print(f"📊 Debug: No formula (hardcoded), using historical value from {baseline_source} = {baseline_val}")
    
    # Get enriched description from lookup (no API call needed!)
    cell_key = f"{sheet}!{cell}"
    enriched_desc = enriched_descriptions.get(cell_key, item.get("description", ""))
    
    # Determine which cells to include
    if "treasury stock - common stock calculation" in desc:
        # Force EQ, ER, and ES for treasury stock rows
        for col in ["EQ", "ER", "ES"]:
            if col in col_period_map:
                cell_ref = f"{col}{row}"
                cell_key_treasury = (sheet, cell_ref)
                if cell_key_treasury in recorded_cells:
                    continue
                recorded_cells.add(cell_key_treasury)

                try:
                    original_val_treasury = wb_values[sheet][cell_ref].value
                except Exception as e:
                    print(f"⚠️ Error reading original value from {sheet}!{cell_ref}: {e}")
                    original_val_treasury = None

                try:
                    formula_obj = wb_formulas[sheet][cell_ref]
                    cell_formula = formula_obj.value if isinstance(formula_obj.value, str) and formula_obj.value.startswith("=") else ""
                except Exception:
                    cell_formula = ""

                # Use the terminal period format if available, otherwise fallback to col_period_map
                period_display = terminal_period_map.get(sheet, {}).get(col, col_period_map.get(col, ""))

                # Use the same enriched description for all treasury stock cells in the same row
                baseline_inputs.append({
                    "sheet": sheet,
                    "cell": cell_ref,
                    "desc": enriched_desc,  # Use pre-computed enriched description
                    "original_value": original_val_treasury,
                    "change_to": baseline_val,
                    "type": "baseline_forcing",
                    "info": f"Forced to {baseline_val} from {baseline_source}",
                    "period": period_display,
                    "formula": cell_formula
                })
    else:
        # Regular terminal: only the terminal cell itself
        terminal_col = re.match(r"[A-Z]+", cell).group(0)
        cell_key_regular = (sheet, cell)
        if cell_key_regular in recorded_cells:
            continue
        recorded_cells.add(cell_key_regular)

        # Get the formula for THIS specific cell, not from the original item
        try:
            formula_obj = wb_formulas[sheet][cell]
            cell_formula = formula_obj.value if isinstance(formula_obj.value, str) and formula_obj.value.startswith("=") else ""
        except Exception:
            cell_formula = ""

        # Use the period from the terminal input directly (this is already in the correct format)
        period_display = item["period"]

        baseline_inputs.append({
            "sheet": sheet,
            "cell": cell,
            "desc": enriched_desc,  # Use pre-computed enriched description
            "original_value": original_val,
            "change_to": baseline_val,
            "type": "baseline_forcing",
            "info": f"Forced to {baseline_val} from {baseline_source}",
            "period": period_display,
            "formula": cell_formula  # Use the cell-specific formula, not the original item's formula
        })

# --- Print result summary ---
# --- Print result summary ---
# ✅ FINAL PATCH: Use the results from the enrichment function
enriched_lookup_for_printing = {}
for result in enriched_results:  # Use 'results' instead of 'enriched_results'
    cell_key = f"{result['sheet']}!{result['cell']}"
    enriched_lookup_for_printing[cell_key] = result['enriched_desc']

print(f"📊 Found {len(enriched_lookup_for_printing)} enriched descriptions for printing")

# Update baseline_inputs with enriched descriptions
updated = 0
for bi in baseline_inputs:
    cell_key = f"{bi['sheet']}!{bi['cell']}"
    if cell_key in enriched_lookup_for_printing:
        bi['desc'] = enriched_lookup_for_printing[cell_key]
        updated += 1

print(f"\n✅ Replaced descriptions in {updated} baseline entries with enriched versions.")

# ✅ Print updated results with ENRICHED descriptions
print("\n📊 Planned Baseline Changes (Terminal):")
print("=" * 150)

for bi in baseline_inputs:
    orig_val_str = f"{bi['original_value']:.4f}" if isinstance(bi['original_value'], (int, float)) else str(bi['original_value'])
    change_to_str = f"{bi['change_to']:.4f}" if isinstance(bi['change_to'], (int, float)) else str(bi['change_to'])
    formula_str = bi.get("formula", "")
    
    # Print with full enriched description (no truncation)
    print(f"Cell: {bi['sheet']}!{bi['cell']:<12} | Period: {bi['period']:<8}")
    print(f"Description: {bi['desc']}")  # Use enriched description
    print(f"Change: {orig_val_str} → {change_to_str} | Info: {bi['info']}")
    if formula_str:
        print(f"Formula: {formula_str}")
    print("-" * 100)

print(f"\n📊 Total baseline changes planned: {len(baseline_inputs)}")

📊 Created enriched description lookup for 49 cells
📊 Found 49 enriched descriptions for printing

✅ Replaced descriptions in 49 baseline entries with enriched versions.

📊 Planned Baseline Changes (Terminal):
Cell: Model!EQ133        | Period: 2Q25E   
Description: Shares Outstanding (mm) (Fully Diluted)
Change: 948.3754 → 948.3754 | Info: Forced to 948.3754128773256 from Model!EP133
Formula: =EP133
----------------------------------------------------------------------------------------------------
Cell: Moon Model!CL8          | Period: 2Q25E   
Description: Net Retail Revenue → % YoY → yoy
Change: 0.0200 → 0.0000 | Info: Forced to 0 from forced_zero_due_to_indicator
Formula: =CK8
----------------------------------------------------------------------------------------------------
Cell: Moon Model!CN9          | Period: 4Q25E   
Description: Net Retail Revenue → % YoY → yoy
Change: 0.0811 → 0.0000 | Info: Forced to 0 from forced_zero_due_to_indicator
-----------------------------------

In [ ]:
import re
import xlwings as xw
import shutil
import time
from datetime import datetime
import os

# === Filter invalid baseline inputs ===
baseline_inputs = [item for item in baseline_inputs if item["change_to"] is not None]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
TEST_FILE = f"test_attribution_{TARGET_SHEET}_{TARGET_CELL}_{timestamp}.xlsx"

print("🚀 Starting simplified attribution analysis...")

# === 1. Copy model ===
shutil.copy2(WORKBOOK_FILE, TEST_FILE)
print(f"📂 Excel file copied: {TEST_FILE}")

# === 2. Open workbook directly (simplified approach) ===
print("📂 Opening Excel workbook...")
try:
    wb = xw.Book(TEST_FILE)
    app = wb.app
    app.visible = True
    app.screen_updating = False
    app.enable_events = False
    app.display_alerts = False
    app.calculation = 'manual'
    print("✅ Excel workbook opened successfully")
except Exception as e:
    print(f"⚠️ Error opening workbook: {e}")
    raise

# === 3. Get original forecast value (before any changes) ===
print("📌 Reading original forecast value...")
original_forecast_value = wb.sheets[TARGET_SHEET].range(TARGET_CELL).value
print(f"📌 Original Forecast Target Value: {original_forecast_value:.4f}")

# === 4. Apply baseline ===
print("🔧 Applying baseline changes...")
changes_applied = 0
changes_verified = 0

for i, item in enumerate(baseline_inputs):
    sheet, cell, change_to = item["sheet"], item["cell"], item["change_to"]
    original_value = item["original_value"]
    
    try:
        rng = wb.sheets[sheet].range(cell)
        
        # Check current value before change
        current_value = rng.value
        
        # Apply change
        rng.formula = None
        rng.value = change_to
        changes_applied += 1
        
        # Verify the change was applied
        new_value = rng.value
        if abs(new_value - change_to) < 1e-10:
            changes_verified += 1
        
        # Show first few changes for debugging
        if i < 5:
            print(f"   {sheet}!{cell}: {original_value:.6f} → {current_value:.6f} → {new_value:.6f} (target: {change_to:.6f})")
        elif i == 5:
            print("   ...")
        
        if i % 10 == 0:  # Progress indicator
            print(f"   Applied {i+1}/{len(baseline_inputs)} changes...")
            
    except Exception as e:
        print(f"⚠️ Error applying baseline to {sheet}!{cell}: {e}")

print(f"📊 Changes applied: {changes_applied}/{len(baseline_inputs)}")
print(f"📊 Changes verified: {changes_verified}/{len(baseline_inputs)}")

# === 4.5. Force calculation after all changes ===
print("\n🔄 Forcing calculation after all baseline changes...")
try:
    app.calculate()
    time.sleep(1)
    # Force a second calculation to ensure all dependencies are updated
    app.calculate()
    time.sleep(1)
    print("✅ Forced calculations completed")
except Exception as e:
    print(f"⚠️ Error during forced calculation: {e}")

# === 5. Calculate baseline ===
print("\n🧮 Calculating baseline...")
for i in range(5):  # Increased attempts
    try:
        print(f"   Calculation attempt {i+1}...")
        app.calculate()
        time.sleep(2)
        
        # Check if target value changed
        current_target = wb.sheets[TARGET_SHEET].range(TARGET_CELL).value
        print(f"   Target after calc {i+1}: {current_target:.6f}")
        
        if abs(current_target - original_forecast_value) > 1e-10:
            print("   ✅ Target value changed, calculation successful")
            break
        else:
            print("   ⚠️ Target unchanged, trying again...")
            
    except Exception as e:
        print(f"⚠️ Excel busy (attempt {i+1}): {e}")
        time.sleep(3)

base_target_value = wb.sheets[TARGET_SHEET].range(TARGET_CELL).value
print(f"📌 Baseline Target Value: {base_target_value:.6f}")

# === 5.5. Diagnostic: Check if any baseline changes are still applied ===
print("\n🔍 Diagnostic: Verifying baseline changes are still applied...")
still_applied = 0
reverted_count = 0

for i, item in enumerate(baseline_inputs[:10]):  # Check first 10 for speed
    sheet, cell, change_to = item["sheet"], item["cell"], item["change_to"]
    original_value = item["original_value"]
    
    try:
        current_value = wb.sheets[sheet].range(cell).value
        if abs(current_value - change_to) < 1e-10:
            still_applied += 1
        elif abs(current_value - original_value) < 1e-10:
            reverted_count += 1
        
        if i < 5:
            print(f"   {sheet}!{cell}: current={current_value:.6f}, should_be={change_to:.6f}, original={original_value:.6f}")
    except Exception as e:
        print(f"⚠️ Error checking {sheet}!{cell}: {e}")

print(f"📊 First 10 cells: {still_applied} still changed, {reverted_count} reverted to original")

# === 5.6. Check if calculation mode is working ===
print(f"\n🔍 Excel calculation mode: {app.calculation}")
if app.calculation != 'manual':
    print("⚠️ Calculation mode is not manual, setting to manual...")
    try:
        app.calculation = 'manual'
        print("✅ Set to manual calculation")
    except Exception as e:
        print(f"⚠️ Could not set manual calculation: {e}")

# === 5.7. If still no change, try more aggressive calculation ===
if abs(base_target_value - original_forecast_value) < 1e-10:
    print("\n🔄 Target still unchanged, trying more aggressive calculation...")
    try:
        # Try calculating specific sheets
        wb.sheets[TARGET_SHEET].calculate()
        time.sleep(1)
        app.calculate()
        time.sleep(2)
        
        # Check again
        final_check = wb.sheets[TARGET_SHEET].range(TARGET_CELL).value
        print(f"📌 Target after aggressive calc: {final_check:.6f}")
        
        if abs(final_check - original_forecast_value) > 1e-10:
            base_target_value = final_check
            print("✅ Aggressive calculation worked!")
        else:
            print("⚠️ Target still unchanged - there may be an issue with the model or inputs")
            
    except Exception as e:
        print(f"⚠️ Error during aggressive calculation: {e}")

# === 6. Group baseline inputs by sheet and row ===
from collections import defaultdict

def extract_row_from_cell(cell):
    """Extract row number from Excel cell reference (e.g., 'A5' -> 5, 'BC123' -> 123)"""
    match = re.search(r'(\d+)', cell)
    return int(match.group(1)) if match else None

def find_common_description_part(descriptions):
    """Find the common part among multiple descriptions"""
    if not descriptions:
        return ""
    if len(descriptions) == 1:
        return descriptions[0]
    
    # Find common prefix
    common_prefix = ""
    min_len = min(len(desc) for desc in descriptions)
    
    for i in range(min_len):
        chars = set(desc[i] for desc in descriptions)
        if len(chars) == 1:
            common_prefix += list(chars)[0]
        else:
            break
    
    # Find common suffix
    common_suffix = ""
    for i in range(1, min_len - len(common_prefix) + 1):
        chars = set(desc[-i] for desc in descriptions)
        if len(chars) == 1:
            common_suffix = list(chars)[0] + common_suffix
        else:
            break
    
    # Combine and clean up
    common_part = (common_prefix + common_suffix).strip()
    
    # If common part is too short or empty, use the first description
    if len(common_part) < 10:
        return descriptions[0]
    
    return common_part

# Group by sheet and row instead of by enriched description
grouped_inputs = defaultdict(list)
for item in baseline_inputs:
    sheet = item["sheet"]
    cell = item["cell"]
    row = extract_row_from_cell(cell)
    
    if row is not None:
        # Group by sheet and row
        group_key = f"{sheet}_row_{row}"
        grouped_inputs[group_key].append(item)
    else:
        # Fallback: group by individual cell if row extraction fails
        group_key = f"{sheet}_{cell}"
        grouped_inputs[group_key].append(item)

print(f"\n📊 Grouped into {len(grouped_inputs)} categories (by sheet and row)")

# === Create final grouped results with common descriptions ===
final_grouped_inputs = {}
for group_key, items in grouped_inputs.items():
    # Extract all descriptions from items in this group
    descriptions = [item['desc'] for item in items]
    
    # Find common description part
    common_desc = find_common_description_part(descriptions)
    
    # Create a more readable group name
    if len(items) == 1:
        final_key = common_desc
    else:
        sheet_name = items[0]['sheet']
        row_num = extract_row_from_cell(items[0]['cell'])
        final_key = f"{common_desc} (Row {row_num})"
    
    final_grouped_inputs[final_key] = items

# === Show grouping summary ===
print("\n📋 Grouping Summary:")
for i, (desc, items) in enumerate(final_grouped_inputs.items(), 1):
    cells_in_group = [f"{item['sheet']}!{item['cell']}" for item in items]
    cells_display = ", ".join(cells_in_group[:3])
    if len(cells_in_group) > 3:
        cells_display += f" (+{len(cells_in_group)-3} more)"
    print(f"  {i:2d}. {desc:<50} | {len(items):2d} cells | {cells_display}")

# === Diagnostic: Check for ungrouped inputs ===
grouped_cells = {f"{item['sheet']}!{item['cell']}" for group in final_grouped_inputs.values() for item in group}
input_cells = {f"{item['sheet']}!{item['cell']}" for item in baseline_inputs}
missing_cells = input_cells - grouped_cells

if missing_cells:
    print(f"\n⚠️ {len(missing_cells)} baseline input cells were not grouped:")
    for cell in sorted(list(missing_cells)[:5]):  # Show first 5
        print(f"  - {cell}")
    if len(missing_cells) > 5:
        print(f"  ... and {len(missing_cells) - 5} more")
else:
    print("\n✅ All baseline inputs were grouped correctly.")

# === 7. Attribution (change-back by group) ===
results = []
total_groups = len(final_grouped_inputs)

for group_idx, (group_desc, items) in enumerate(final_grouped_inputs.items(), 1):
    print(f"\n🔄 [{group_idx}/{total_groups}] Changing back group: {group_desc}")

    target_before = wb.sheets[TARGET_SHEET].range(TARGET_CELL).value

    # Revert each cell in this group to its original forecast value
    for item in items:
        sheet, cell, original_val = item["sheet"], item["cell"], item["original_value"]
        try:
            rng = wb.sheets[sheet].range(cell)
            rng.formula = None
            rng.value = original_val
        except Exception as e:
            print(f"⚠️ Error reverting {sheet}!{cell}: {e}")

    # Calculate after reverting this group
    for j in range(2):  # Reduced attempts for speed
        try:
            app.calculate()
            time.sleep(1)
            break
        except Exception as e:
            print(f"⚠️ Excel busy (attempt {j+1}): {e}")
            time.sleep(2)

    new_target_value = wb.sheets[TARGET_SHEET].range(TARGET_CELL).value
    delta = new_target_value - base_target_value

    print(f"   🎯 Target: {target_before:.4f} → {new_target_value:.4f} (Δ = {delta:+.4f})")

    results.append({
        "description": group_desc,  # Use the common description for the row group
        "cells": [f"{item['sheet']}!{item['cell']}" for item in items],
        "delta": delta
    })

    # Restore cells in this group back to baseline
    for item in items:
        try:
            rng = wb.sheets[item["sheet"]].range(item["cell"])
            rng.value = item["change_to"]
        except Exception as e:
            print(f"⚠️ Error restoring {item['sheet']}!{item['cell']}: {e}")

# === 8. Normalize results ===
total_delta = original_forecast_value - base_target_value

print(f"\n✅ Total groups analyzed: {len(results)}")
print(f"📉 Total Δ Target: {total_delta:+.4f}")

for r in results:
    if abs(total_delta) > 1e-6:
        r["normalized_pct"] = (r["delta"] / total_delta) * 100
    else:
        r["normalized_pct"] = 0.0

# === Add residual if needed ===
sum_of_group_deltas = sum(r["delta"] for r in results)
residual_delta = total_delta - sum_of_group_deltas

if abs(residual_delta) > 1e-6:
    residual_pct = residual_delta / total_delta * 100 if abs(total_delta) > 1e-6 else 0
    results.append({
        "description": "Interrelated effects",
        "cells": [],
        "delta": residual_delta,
        "normalized_pct": residual_pct
    })
    print(f"\n⚠️ Added Interrelated effects: Δ={residual_delta:+.4f}, % of Total={residual_pct:+.2f}%")

# === 9. Close Excel ===
print("\n🔄 Closing workbook...")
try:
    wb.close()
    app.enable_events = True
    app.display_alerts = True
    # Don't call app.quit() - this leaves Excel open with other files
    print("✅ Workbook closed, Excel application remains open.")
except Exception as e:
    print(f"⚠️ Error closing workbook: {e}")

# === 10. Final report ===
print("\n📊 Final Attribution Results (% of total Δ Target):")
print(f"{'Row-Based Group Description':<60} | {'% of Total':>10} | {'Δ Target':>12} | {'Cells'}")
print("=" * 120)

for r in sorted(results, key=lambda x: abs(x['delta']), reverse=True):
    cell_list = ", ".join(r['cells'][:3])  # Show first 3 cells
    if len(r['cells']) > 3:
        cell_list += f" (+{len(r['cells'])-3} more)"
    
    print(
        f"{r['description']:<60} | {r['normalized_pct']:+8.2f}% | "
        f"{r['delta']:+10.4f} | {cell_list}"
    )

# === 11. Check sum ===
print(f"\n✅ Sum of all deltas: {sum_of_group_deltas:+.4f}")
print(f"📊 Total change: {total_delta:+.4f}")
print(f"🎯 Residual: {residual_delta:+.4f}")

print(f"\n🎉 Attribution analysis complete!")

🚀 Starting simplified attribution analysis...
📂 Excel file copied: test_attribution_Model_ET137_20250728_075911.xlsx
📂 Opening Excel workbook...
✅ Excel workbook opened successfully
📌 Reading original forecast value...
📌 Original Forecast Target Value: 1.2177
🔧 Applying baseline changes...
   Model!EQ133: 948.375413 → 948.375413 → 948.375413 (target: 948.375413)
   Applied 1/50 changes...
   Revenue!EP40: 2738.959608 → 2738.959608 → 2738.959608 (target: 2738.959608)
   Moon Model!CL8: 0.020000 → 0.020000 → 0.000000 (target: 0.000000)
   Moon Model!CN9: 0.081058 → 0.081058 → 0.000000 (target: 0.000000)
   Moon Model!CL9: 0.097137 → 0.097137 → 0.000000 (target: 0.000000)
   ...
   Applied 11/50 changes...
   Applied 21/50 changes...
   Applied 31/50 changes...
   Applied 41/50 changes...
📊 Changes applied: 50/50
📊 Changes verified: 50/50

🔄 Forcing calculation after all baseline changes...
✅ Forced calculations completed

🧮 Calculating baseline...
   Calculation attempt 1...
   Target af

In [15]:
print("\n📊 Top Attribution Results (% of total Δ):")
top_results = sorted(results, key=lambda r: abs(r["delta"]), reverse=True)[:10]
for r in top_results:
    print(f"{r['description']:<40} | % of Total: {r['normalized_pct']:+.2f}%")


📊 Top Attribution Results (% of total Δ):
(-) Retail COGS → (=) Gross Profit, Retail → % Margin (Row 34) | % of Total: +87.84%
(=) Net Income → (-) Taxes → % Tax Rate, Eff. (Row 67) | % of Total: -48.34%
Net Retail Revenue → % YoY → yoy (Row 9) | % of Total: +34.18%
(+/-) Working Capital Changes → Inventories → Inventory, Days of COGS (Row 104) | % of Total: +9.79%
Financial Revenue → CIRC → % of Avg Cash Balance (Row 148) | % of Total: +9.43%
(+/-) Working Capital Changes → A/R → A/R, Days of Revs (Row 103) | % of Total: +7.95%
Net Retail Revenue → % YoY → yoy         | % of Total: +7.95%
Interrelated effects                     | % of Total: -6.42%
CFF → Dividends Paid → % DPO             | % of Total: -6.32%
SG&A item 1 → (-) Opex → % of Sales (Row 36) | % of Total: +4.13%


In [16]:
import pandas as pd

def format_number(value, sig_digits=2):
    """Format number to 2-3 significant digits"""
    if value is None or value == "":
        return ""
    try:
        num = float(value)
        if num == 0:
            return "0"
        # Round to significant digits
        from math import log10, floor
        rounded = round(num, -int(floor(log10(abs(num)))) + (sig_digits - 1))
        # Format nicely
        if abs(rounded) >= 1:
            return f"{rounded:.2f}".rstrip('0').rstrip('.')
        else:
            return f"{rounded:.3f}".rstrip('0').rstrip('.')
    except:
        return str(value)

# Build a lookup dictionary from baseline_inputs for fast access
cell_lookup = {}
for bi in baseline_inputs:
    key = f"{bi['sheet']}!{bi['cell']}"
    cell_lookup[key] = {
        "original": format_number(bi.get("original_value")),
        "change_to": format_number(bi.get("change_to")),
        "formula": bi.get("formula", ""),
        "period": bi.get("period", ""),
        "info": bi.get("info", "")
    }

# Define thresholds for meaningful effects
MIN_DELTA_THRESHOLD = 1e-6  # Minimum absolute delta to be considered meaningful
MIN_PCT_THRESHOLD = 0.01    # Minimum absolute percentage to be considered meaningful (0.01%)

print(f"📊 Filtering results with thresholds:")
print(f"   - Minimum |Δ Target|: {MIN_DELTA_THRESHOLD}")
print(f"   - Minimum |% of Total|: {MIN_PCT_THRESHOLD}%")

# Filter results to exclude zero/negligible effects
meaningful_results = []
excluded_count = 0
excluded_total_delta = 0

for r in results:
    abs_delta = abs(r['delta'])
    abs_pct = abs(r['normalized_pct'])
    
    # Keep if delta or percentage is above threshold
    if abs_delta >= MIN_DELTA_THRESHOLD and abs_pct >= MIN_PCT_THRESHOLD:
        meaningful_results.append(r)
    else:
        excluded_count += 1
        excluded_total_delta += r['delta']
        print(f"   ❌ Excluded: '{r['description'][:40]}...' (Δ={r['delta']:+.6f}, %={r['normalized_pct']:+.3f}%)")

print(f"\n📈 Filtering Summary:")
print(f"   ✅ Meaningful results: {len(meaningful_results)}")
print(f"   ❌ Excluded (zero/negligible): {excluded_count}")
print(f"   📉 Total excluded delta: {excluded_total_delta:+.6f}")

# Add excluded effects as a summary line if there are any
if excluded_count > 0 and abs(excluded_total_delta) >= MIN_DELTA_THRESHOLD:
    # Calculate percentage of total for excluded effects
    total_delta = original_forecast_value - base_target_value
    excluded_pct = (excluded_total_delta / total_delta * 100) if abs(total_delta) > 1e-6 else 0
    
    meaningful_results.append({
        "description": f"Minor effects (< {MIN_PCT_THRESHOLD}% each, {excluded_count} items)",
        "cells": [],
        "delta": excluded_total_delta,
        "normalized_pct": excluded_pct
    })
    print(f"   📝 Added summary line for minor effects: Δ={excluded_total_delta:+.6f}, %={excluded_pct:+.2f}%")

# Create export data by combining attribution summary with baseline details
export_data = []
for r in meaningful_results:
    details = []
    for cell in r['cells']:
        meta = cell_lookup.get(cell, {})
        # Clean, spaced format
        details.append(
            f"{cell} | Period: {meta.get('period')} | "
            f"Original: {meta.get('original')} | "
            f"Change To: {meta.get('change_to')} | "
            f"Formula: {meta.get('formula')} | "
            f"Info: {meta.get('info')}"
        )

    # Parse numeric percent value for sorting
    numeric_pct = float(r['normalized_pct'])

    export_data.append({
        "Description": r["description"],
        "% of Total": f"{numeric_pct:+.2f}%",
        "Δ Target": f"{r['delta']:+.4f}",
        "Affected Cells": ", ".join(r["cells"]) if r["cells"] else "Multiple minor effects",
        "Details": "\n".join(details) if details else "Summary of multiple small effects",
        "_sort_key": abs(numeric_pct)
    })

# Sort by absolute % contribution descending
export_data_sorted = sorted(export_data, key=lambda x: -x["_sort_key"])

# Remove helper key and export to Excel
for row in export_data_sorted:
    del row["_sort_key"]

df = pd.DataFrame(export_data_sorted)

# Create filename with meaningful count
meaningful_count = len([r for r in meaningful_results if r["cells"]])  # Exclude summary lines
filename = f"meaningful_attribution_results_{meaningful_count}_effects.xlsx"

# Export to Excel with formatting
try:
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Attribution Results', index=False)
        
        # Get the workbook and worksheet for formatting
        workbook = writer.book
        worksheet = writer.sheets['Attribution Results']
        
        # Auto-adjust column widths
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            
            # Set column width with some padding
            adjusted_width = min(max_length + 2, 100)  # Cap at 100 characters
            worksheet.column_dimensions[column_letter].width = adjusted_width
        
        # Make header row bold
        from openpyxl.styles import Font
        for cell in worksheet[1]:
            cell.font = Font(bold=True)
    
    print(f"\n✅ Exported {len(export_data_sorted)} meaningful results to {filename}")
    print(f"📊 Results include {meaningful_count} actual effects plus any summary lines")
    
except Exception as e:
    # Fallback to basic export if formatting fails
    df.to_excel(filename, index=False)
    print(f"\n✅ Exported {len(export_data_sorted)} meaningful results to {filename} (basic format)")
    print(f"⚠️ Advanced formatting failed: {e}")


# Show a preview of the top results
print(f"\n🔝 Top 5 Meaningful Results:")
for i, row in enumerate(export_data_sorted[:5], 1):
    print(f"  {i}. {row['Description'][:50]:<50} | {row['% of Total']:>8} | {row['Δ Target']:>10}")

# Verification: Check that we haven't lost significant effects
total_meaningful_delta = sum(r['delta'] for r in meaningful_results)
total_original_delta = sum(r['delta'] for r in results)
coverage = (total_meaningful_delta / total_original_delta * 100) if abs(total_original_delta) > 1e-6 else 100

print(f"\n🎯 Verification:")
print(f"   Original total effects: {len(results)}")
print(f"   Meaningful effects kept: {len(meaningful_results)}")
print(f"   Delta coverage: {coverage:.1f}% of original total")

📊 Filtering results with thresholds:
   - Minimum |Δ Target|: 1e-06
   - Minimum |% of Total|: 0.01%
   ❌ Excluded: 'Shares Outstanding (mm) (Fully Diluted)...' (Δ=+0.000000, %=+0.000%)
   ❌ Excluded: 'Revenue item 1...' (Δ=+0.000000, %=+0.000%)
   ❌ Excluded: 'D&A → Memo: D&A ex RoU → % of Sales (Row...' (Δ=+0.000000, %=+0.000%)
   ❌ Excluded: 'D&A → Memo: D&A RoU → % of Sales (Row 51...' (Δ=+0.000000, %=+0.000%)
   ❌ Excluded: 'EoP Cash and Equivalents → BoP Cash and ...' (Δ=+0.000000, %=+0.000%)
   ❌ Excluded: 'Interest Expense → EoP Debt → ST Debts...' (Δ=+0.000000, %=+0.000%)
   ❌ Excluded: 'Interest Expense → EoP Debt → LT Debts...' (Δ=+0.000000, %=+0.000%)
   ❌ Excluded: 'Interest Expense → Financing Leases → ST...' (Δ=+0.000000, %=+0.000%)
   ❌ Excluded: 'Interest Expense → Financing Leases → LT...' (Δ=+0.000000, %=+0.000%)
   ❌ Excluded: 'Financial Expense → Interest Expense → %...' (Δ=+0.000000, %=+0.000%)
   ❌ Excluded: '(+/-) Working Capital Changes → A/R → Ro...' (Δ=+0.000

In [32]:
from openpyxl import Workbook 
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font
import numpy as np
import pandas as pd

# At the very beginning of your code, change this line:
TOP_N = 5  # Changed from 6 to 5
BLANK_ROWS_AFTER_TABLE = 5

# === Format numbers ===
def format_number_custom(x):
    try:
        x = float(x)
        if abs(x) >= 100:
            return f"{x:.0f}"
        else:
            return f"{x:.2f}"
    except:
        return x

def format_pct(x):
    try:
        val = float(x)
        if np.isnan(val):
            return ""
        return f"{val:+.1f}%"  # This is already correct - 1 decimal place
    except:
        return ""

# === Prepare export rows ===
sorted_results = sorted(meaningful_results, key=lambda x: abs(x["normalized_pct"]), reverse=True)
top_results = sorted_results[:TOP_N]  # This will now take the top 5 instead of top 6
grouped_results = sorted_results[TOP_N:]  # Everything after the top 5 goes here

# Separate interrelated effects (set aside completely)
interrelated = next((r for r in grouped_results if r["description"] == "Interrelated effects"), None)
if interrelated:
    grouped_results = [r for r in grouped_results if r["description"] != "Interrelated effects"]

    
# === Build export rows ===
def build_export_row(r):
    details = []
    for cell in r.get("cells", []):
        meta = cell_lookup.get(cell, {})
        details.append(
            f"{cell} | Period: {meta.get('period')} | "
            f"Original: {meta.get('original')} | "
            f"Change To: {meta.get('change_to')} | "
            f"Formula: {meta.get('formula')} | "
            f"Info: {meta.get('info')}"
        )
    return {
        "Description": r["description"],
        "% of Total": format_pct(r["normalized_pct"]),
        "Δ Target": format_number_custom(r["delta"]),
        "Callable Link": "",
        "Affected Cells": ", ".join(r.get("cells", [])) if r.get("cells") else "Multiple minor effects",
        "Details": "\n".join(details) if details else "Summary of multiple small effects"
    }

export_data = [build_export_row(r) for r in top_results]

# Add grouped lower impact drivers (WITHOUT Interrelated Effects)
if grouped_results:
    export_data.append({
        "Description": f"Lower Impact Drivers (n = {len(grouped_results)})",
        "% of Total": format_pct(sum(r['normalized_pct'] for r in grouped_results)),
        "Δ Target": format_number_custom(sum(r['delta'] for r in grouped_results)),
        "Callable Link": "",
        "Affected Cells": "",
        "Details": ""
        
    })
    export_data.extend(build_export_row(r) for r in grouped_results)

# Add Interrelated Effects at the bottom (just before Sum row)
if interrelated:
    export_data.append(build_export_row(interrelated))

# Sum row (exclude Interrelated Effects)
sum_delta = sum(r["delta"] for r in top_results + grouped_results)
export_data.append({
    "Description": "Sum",
    "% of Total": "100.0%",  # Change this to have only 1 decimal place
    "Δ Target": format_number_custom(sum_delta),
    "Callable Link": "",
    "Affected Cells": "",
    "Details": ""
})

# === Build Excel Workbook ===
wb = Workbook()
ws = wb.active
ws.title = "Attribution Results"

# === EPS Summary Block ===
historical_actual = wb_values[TARGET_SHEET][BASELINE_COMPARISON_CELL].value
current_forecasted = wb_values[TARGET_SHEET][TARGET_CELL].value  # or use `actual_2025` if already defined
original_name = os.path.splitext(WORKBOOK_FILE)[0]

ws.append([f"Result for {original_name}"])
ws["A1"].font = Font(bold=True)
ws.append(["Historical Actual Value", format_number_custom(historical_actual)])
ws.append(["Baseline Forecasted Value", format_number_custom(base_target_value)])
ws.append(["Current Forecasted Value", format_number_custom(current_forecasted)])
ws.append(["Change in Target", format_number_custom(original_forecast_value - base_target_value)])
ws.append([])

# === Export Table ===
ws.freeze_panes = "A6"
start_row = ws.max_row + 1
df_final = pd.DataFrame(export_data)
for row in dataframe_to_rows(df_final, index=False, header=True):
    ws.append(row)

# === Group lower impact rows (add outlining/collapsing functionality) ===
group_header_row = None
sum_row = None
for i, row in enumerate(ws.iter_rows(min_row=start_row, max_row=ws.max_row), start=start_row):
    val = row[0].value
    if val and isinstance(val, str):
        if val.startswith("Lower Impact Drivers"):
            group_header_row = i
        elif val == "Sum":
            sum_row = i

for cell in ws[sum_row]:
    cell.font = Font(bold=True)

if group_header_row and sum_row:
    # Find Interrelated Effects row (should be just before Sum)
    interrelated_row = None
    for i in range(sum_row - 1, group_header_row, -1):
        val = ws.cell(row=i, column=1).value
        if val and isinstance(val, str) and "Interrelated Effects" in val:
            interrelated_row = i
            break
    
    # Group only the Lower Impact Drivers detail rows (not Interrelated Effects)
    group_end_row = interrelated_row - 1 if interrelated_row else sum_row - 1
    
    for r in range(group_header_row + 1, group_end_row + 1):
        ws.row_dimensions[r].outlineLevel = 1
        ws.row_dimensions[r].hidden = True
    ws.sheet_properties.outlinePr.summaryBelow = True

# === Add blank rows ===
for _ in range(BLANK_ROWS_AFTER_TABLE):
    ws.append([])

# === EPS Impact Summary Block ===
# === EPS Impact Summary Block ===
normalized_vals = [
    abs(float(r["% of Total"].strip('%')))
    for r in export_data if r["Description"] not in ("Sum",) and not r["Description"].startswith("Lower Impact")
]
q75, q50, q25 = np.percentile(normalized_vals, [75, 50, 25]) if normalized_vals else (0, 0, 0)

impact_summary = {
    "High Impact (Top 25%)": [],
    "Medium - High (Next 25%)": [],
    "Medium - Low (Next 25%)": [],
    "Low Impact (Bottom 25%)": []
}
for r in export_data:
    if r["Description"] in ("Sum",) or r["Description"].startswith("Lower Impact"):
        continue
    val = abs(float(r["% of Total"].strip('%')))
    if val >= q75:
        impact_summary["High Impact (Top 25%)"].append(r["Description"])
    elif val >= q50:
        impact_summary["Medium - High (Next 25%)"].append(r["Description"])
    elif val >= q25:
        impact_summary["Medium - Low (Next 25%)"].append(r["Description"])
    else:
        impact_summary["Low Impact (Bottom 25%)"].append(r["Description"])

ws.append(["EPS Impact Summary"])
summary_title_row = ws.max_row
for cell in ws[summary_title_row]:
    cell.font = Font(bold=True)

for level, descriptions in impact_summary.items():
    # Add the category header in column A
    ws.append([level])
    
    # Add count in column A, one row under the category
    ws.append([f"Count: {len(descriptions)}"])
    
    # Add descriptions in column B
    for desc in descriptions:
        ws.append(["", desc])  # Empty column A, description in column B
    
    # Add blank row between sections
    ws.append([])

# === Bold header row ===
for cell in ws[f"A{start_row}":f"A{start_row}"][0]:
    cell.font = Font(bold=True)

# === Column Widths ===
column_widths = {
    "A": 72,
    "B": 12,
    "C": 12,
    "D": 8,
    "E": 20,
    "F": 24
}
for col_letter, width in column_widths.items():
    ws.column_dimensions[col_letter].width = width

####################### XLWINGS SECTION WITH VBA BUTTONS ##############################
import xlwings as xw
import os
import shutil
from datetime import datetime

# Configuration
TEMPLATE_PATH = "template_with_macro.xlsm"
FINAL_FILENAME = f"attribution_test_{original_name}.xlsm"

print(f"🚀 Starting VBA-enabled Excel file generation...")

# Step 1: Save the openpyxl workbook as temporary xlsx file
temp_xlsx = f"temp_attribution_{timestamp}.xlsx"
wb.save(temp_xlsx)
print(f"💾 Saved temporary file: {temp_xlsx}")

# Step 2: Copy template and prepare final file
if os.path.exists(FINAL_FILENAME):
    try:
        os.remove(FINAL_FILENAME)
        print("🗑️ Removed existing output file")
    except PermissionError:
        print(f"❌ Cannot remove {FINAL_FILENAME} - file may be open")
        exit(1)

shutil.copy(TEMPLATE_PATH, FINAL_FILENAME)
print(f"✅ Copied macro template to: {FINAL_FILENAME}")

# Step 3: Open both workbooks with xlwings
try:
    app = xw.App(visible=False)  # Set to True for debugging
    
    # Open temporary xlsx file (source data)
    temp_book = app.books.open(os.path.abspath(temp_xlsx))
    temp_sheet = temp_book.sheets["Attribution Results"]
    
    # Open macro-enabled xlsm file (destination)
    final_book = app.books.open(os.path.abspath(FINAL_FILENAME))
    final_sheet = final_book.sheets["Attribution Results"]
    
    print("📖 Opened both workbooks successfully")
    
except Exception as e:
    print(f"❌ Error opening workbooks: {e}")
    if 'app' in locals():
        app.quit()
    exit(1)

try:
    # Step 4: Copy all data from temp workbook to final workbook
    used_range = temp_sheet.used_range
    if used_range is not None:
        # Copy all data at once
        data = used_range.value
        final_sheet.range("A1").value = data
        
        print("📊 Copied all data to macro-enabled workbook")
    
    # Step 5: Set original workbook filename in Z1
    final_sheet.range("Z1").value = WORKBOOK_FILE
    print(f"🏷️ Set original filename: {WORKBOOK_FILE}")
    
    # Step 6: Find the data table start row and sum row IN THE FINAL WORKBOOK
    # Look for the main table headers (after EPS Summary)
    DATA_START_ROW = None
    sum_row_xlwings = None
    eps_summary_row = None
    
    # Read column A to find table structure - expand search range
    col_a_range = final_sheet.range("A1:A100")  # Increased search range
    col_a_values = col_a_range.value
    
    for i, value in enumerate(col_a_values):
        if isinstance(value, str):
            excel_row = i + 1  # Excel is 1-indexed
            if value == "Description":  # Table header
                DATA_START_ROW = excel_row
                print(f"🎯 Found table header at row {excel_row}")
            elif value.strip().lower() == "sum":
                sum_row_xlwings = excel_row
                print(f"🎯 Found Sum row at row {excel_row}")
            elif value == "EPS Impact Summary":
                eps_summary_row = excel_row
                print(f"🎯 Found EPS Impact Summary at row {excel_row}")
    
    if DATA_START_ROW is None:
        print("⚠️ Could not find table header 'Description'")
        DATA_START_ROW = 8  # Fallback
    
    if sum_row_xlwings is None:
        print("⚠️ Could not find Sum row")
        sum_row_xlwings = DATA_START_ROW + len(export_data)
    
    print(f"🎯 Table starts at row {DATA_START_ROW}, Sum at row {sum_row_xlwings}")
    
    # === Apply Formatting in Final Workbook ===
    print("🎨 Applying formatting to final workbook...")
    
    # Bold title row (Row 1)
    final_sheet.range("A1").font.bold = True
    print("✅ Applied bold to title row")
    
    # Bold table header row (found dynamically)
    if DATA_START_ROW:
        final_sheet.range(f"{DATA_START_ROW}:{DATA_START_ROW}").font.bold = True
        print(f"✅ Applied bold to header row {DATA_START_ROW}")
    
    # Bold Sum row (if found)
    if sum_row_xlwings:
        final_sheet.range(f"{sum_row_xlwings}:{sum_row_xlwings}").font.bold = True
        print(f"✅ Applied bold to Sum row {sum_row_xlwings}")
    
    # Bold EPS Impact Summary title
    if eps_summary_row:
        final_sheet.range(f"{eps_summary_row}:{eps_summary_row}").font.bold = True
        print(f"✅ Applied bold to EPS Impact Summary row {eps_summary_row}")
    else:
        # Fallback: search again with different approach
        for i in range(1, 100):
            try:
                cell_value = final_sheet.range(f"A{i}").value
                if cell_value == "EPS Impact Summary":
                    final_sheet.range(f"{i}:{i}").font.bold = True
                    print(f"✅ Applied bold to EPS Impact Summary row {i} (fallback)")
                    break
            except:
                continue
    
    # Italicize rows for special categories
    italics_applied = 0
    search_start = DATA_START_ROW if DATA_START_ROW else 1
    
    for i in range(search_start, 100):  # Search reasonable range
        try:
            val = final_sheet.range(f"A{i}").value
            if isinstance(val, str):
                should_italicize = False
                
                # Check for Lower Impact Drivers or Interrelated effects
                if val.startswith("Lower Impact Drivers") or "Interrelated effects" in val:
                    should_italicize = True
                
                # Check for impact level categories
                elif val.strip() in [
                    "High Impact (Top 25%)",
                    "Medium - High (Next 25%)",
                    "Medium - Low (Next 25%)",
                    "Low Impact (Bottom 25%)"
                ]:
                    should_italicize = True
                
                if should_italicize:
                    final_sheet.range(f"{i}:{i}").font.italic = True
                    italics_applied += 1
                    print(f"  ✅ Applied italics to row {i}: '{val[:50]}...'")
                    
        except Exception as e:
            # Stop searching when we hit empty rows or errors
            if "out of range" in str(e).lower():
                break
            continue
    
    print(f"✅ Applied italics to {italics_applied} rows")
    


    # Step 7: Read affected cells column and create buttons
    button_start_row = DATA_START_ROW + 1  # Skip header
    button_end_row = sum_row_xlwings - 1  # Don't include Sum row
    
    if button_end_row >= button_start_row:
        # Read affected cells column (Column E) in the range
        affected_cells_range = final_sheet.range(f"E{button_start_row}:E{button_end_row}")
        affected_cells_values = affected_cells_range.value
        
        # Ensure it's a list even if single row
        if not isinstance(affected_cells_values, list):
            affected_cells_values = [affected_cells_values]
        
        buttons_created = 0
        
        # Create buttons for valid rows
        for i, affected_cells in enumerate(affected_cells_values):
            current_row = button_start_row + i
            
            # Skip if no affected cells or invalid format
            if not affected_cells or not isinstance(affected_cells, str):
                print(f"  ⏭️ Row {current_row}: No affected cells")
                continue
                
            if "!" not in affected_cells:
                print(f"  ⏭️ Row {current_row}: No sheet reference")
                continue
                
            if "multiple minor effects" in affected_cells.lower():
                print(f"  ⏭️ Row {current_row}: Multiple minor effects")
                continue
            
            # Check if button already exists
            button_name = f"btn_{current_row}"
            button_exists = False
            try:
                existing_button = final_sheet.api.Buttons(button_name)
                button_exists = True
                print(f"  ♻️ Row {current_row}: Button already exists")
            except:
                button_exists = False
            
            if not button_exists:
                # ✅ Create the button
                try:
                    target_cell = final_sheet.range(f"D{current_row}")
                    left = target_cell.left
                    top = target_cell.top
                    width = 45
                    height = 15

                    # Extract button label: first cell reference without sheet
                    # E.g., "Sheet1!B2, Sheet2!C4" → "B2"
                    first_ref = affected_cells.split(',')[0].strip()  # Take first entry
                    if '!' in first_ref:
                        first_ref = first_ref.split('!')[1]  # Remove sheet name

                    button = final_sheet.api.Buttons().Add(left, top, width, height)
                    button.Text = first_ref  # Set the extracted reference as label
                    button.Name = button_name
                    button.OnAction = "GoToAffectedCell"

                    buttons_created += 1

                    print(f"  ✅ Row {current_row}: Created button for '{affected_cells[:50]}...'")
                    
                except Exception as e:
                    print(f"  ❌ Row {current_row}: Error creating button - {e}")
        
        print(f"🎯 Created {buttons_created} buttons total")
    
    # Step 8: Apply grouping/outlining to Lower Impact Drivers
    print("🔍 Applying grouping to Lower Impact Drivers...")
    
    # Find rows for grouping in the xlwings workbook
    group_header_row = None
    interrelated_row = None
    
    # Use the already read column A values
    for i, value in enumerate(col_a_values):
        if isinstance(value, str):
            excel_row = i + 1  # Excel is 1-indexed
            
            if value.startswith("Lower Impact Drivers"):
                group_header_row = excel_row
                print(f"    ✅ Found group header at row {excel_row}")
            elif "Interrelated effects" in value:
                interrelated_row = excel_row
                print(f"    ✅ Found interrelated at row {excel_row}")
    
    # Apply grouping if we found the necessary rows
    if group_header_row and sum_row_xlwings:
        group_start_row = group_header_row + 1
        group_end_row = interrelated_row - 1 if interrelated_row else sum_row_xlwings - 1
        
        print(f"🎯 Grouping rows {group_start_row} to {group_end_row}")
        
        if group_start_row <= group_end_row:
            try:
                # Select the range to group
                group_range = final_sheet.range(f"{group_start_row}:{group_end_row}")
                
                # Apply grouping
                group_range.api.Group()
                print(f"✅ Successfully grouped rows {group_start_row}-{group_end_row}")
                
                # Collapse the group (this is the correct way)
                try:
                    # Set outline level to collapsed state
                    final_sheet.api.Outline.ShowLevels(RowLevels=1)
                    print("✅ Successfully collapsed the group")
                except Exception as collapse_error:
                    print(f"⚠️ Group created but couldn't collapse: {collapse_error}")
                
            except Exception as e:
                print(f"❌ Error applying grouping: {e}")
        else:
            print("⚠️ No valid rows to group")
    else:
        print("⚠️ Could not find required rows for grouping")
        print(f"   group_header_row: {group_header_row}")
        print(f"   sum_row_xlwings: {sum_row_xlwings}")
    
    # Step 9: Set column widths in the xlwings workbook
    column_widths = {
        "A": 72,
        "B": 12, 
        "C": 12,
        "D": 12,
        "E": 32,
        "F": 24
    }
    
    for col_letter, width in column_widths.items():
        final_sheet.range(f"{col_letter}:{col_letter}").column_width = width
    
    print("📏 Applied column widths to final workbook")

    # Step 10: Save and close
    final_book.save()
    print("💾 Saved macro-enabled workbook with all formatting applied")
    
except Exception as e:
    print(f"❌ Error during processing: {e}")
    import traceback
    traceback.print_exc()

finally:
    # Always close Excel
    try:
        temp_book.close()
        final_book.close()
        app.quit()
        print("🔒 Closed Excel application")
    except:
        pass
    
    # Clean up temporary file
    try:
        if os.path.exists(temp_xlsx):
            os.remove(temp_xlsx)
            print("🗑️ Cleaned up temporary file")
    except:
        pass

print(f"\n✅ COMPLETED: {FINAL_FILENAME}")
print("📋 Usage Instructions:")
print("1. Ensure your original Excel file is open")
print(f"2. Open {FINAL_FILENAME}")
print("3. Click any 'Button' in the Callable Link column")
print("4. VBA will jump to the original file and highlight cells")
print(f"5. Original attribution count: {meaningful_count} effects")

🚀 Starting VBA-enabled Excel file generation...
💾 Saved temporary file: temp_attribution_20250724_160317.xlsx
🗑️ Removed existing output file
✅ Copied macro template to: attribution_test_Sea.xlsm
📖 Opened both workbooks successfully
📊 Copied all data to macro-enabled workbook
🏷️ Set original filename: Sea.xlsx
🎯 Found table header at row 7
🎯 Found Sum row at row 32
🎯 Found EPS Impact Summary at row 38
🎯 Table starts at row 7, Sum at row 32
🎨 Applying formatting to final workbook...
✅ Applied bold to title row
✅ Applied bold to header row 7
✅ Applied bold to Sum row 32
✅ Applied bold to EPS Impact Summary row 38
  ✅ Applied italics to row 13: 'Lower Impact Drivers (n = 17)...'
  ✅ Applied italics to row 31: 'Interrelated effects...'
  ✅ Applied italics to row 39: 'High Impact (Top 25%)...'
  ✅ Applied italics to row 48: 'Medium - High (Next 25%)...'
  ✅ Applied italics to row 58: 'Medium - Low (Next 25%)...'
  ✅ Applied italics to row 67: 'Low Impact (Bottom 25%)...'
✅ Applied italics t

In [ ]:
import os
import subprocess
import time

# Kill all Excel processes
print("🔄 Closing all Excel processes...")
try:
    subprocess.run(["taskkill", "/F", "/IM", "excel.exe"], 
                   capture_output=True, check=False)
    subprocess.run(["taskkill", "/F", "/IM", "xlwings.exe"], 
                   capture_output=True, check=False)
    time.sleep(2)
    print("✅ Excel processes closed")
except Exception as e:
    print(f"⚠️ Error closing Excel: {e}")

# Your existing code starts here...

🔄 Closing all Excel processes...
✅ Excel processes closed


: 

In [ ]:
    debug_print("✅ Attribution analysis section completed!")

        debug_print("🔍 DEBUG: About to write file...")
        debug_print(f"🔍 DEBUG: output_lines has {len(output_lines)} lines")
        debug_print(f"🔍 DEBUG: debug_output has {len(debug_output)} lines")

        try:
            debug_print("📝 Attempting to write to ARG_CONFIRMATION.txt...")
            #result_file = "C:/Users/pli/Desktop/ARG_CONFIRMATION.txt"
            result_file = "C:/Users/pli/Desktop/ARG_CONFIRMATION.txt"
            debug_print(f"🔍 DEBUG: Writing to {result_file}")

            with open(result_file, "w", encoding="utf-8") as f:
                f.write("🎯 ATTRIBUTION ANALYSIS RESULTS\n")
                f.write("="*60 + "\n")
                for line in debug_output:
                    f.write(f"{line}\n")
                f.write("\n" + "="*60 + "\n")
                for line in output_lines:
                    f.write(f"{line}\n")


            debug_print(f"✅ Results written to {result_file}")
            import os
            if os.path.exists(result_file):
                debug_print(f"✅ File confirmed to exist at {result_file}")
            else:
                debug_print(f"❌ File does not exist at {result_file}")
                
        except Exception as e:
            debug_print(f"❌ Error writing results: {str(e)}")
            import traceback
            debug_print(f"❌ Full traceback: {traceback.format_exc()}")